# LeetCode 1165 — Single-Row Keyboard bằng Python

**Link đề:** https://leetcode.com/problems/single-row-keyboard/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def calculateTime(self, keyboard: str, word: str) -> int:
        ...
```

## Mục tiêu học
1. Mô hình hóa bài toán "ngón tay di chuyển trên một hàng phím": chi phí = **khoảng cách giữa hai vị trí** `|i - j|`.
2. Baseline dùng `keyboard.index(ch)` và hiểu vì sao mỗi lần tra tốn `O(26)`.
3. Kỹ thuật **bảng tra vị trí** (position map) bằng `dict` / dict comprehension / mảng 26 → tra `O(1)`.
4. Mẫu "**biến trạng thái `prev`**" — nhớ vị trí trước để tính chi phí bước kế tiếp.
5. Trace, chứng minh, test, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và ví dụ

Có một bàn phím đặc biệt với **tất cả phím nằm trên một hàng**. Chuỗi `keyboard` dài 26 cho biết bố cục: chữ `keyboard[i]` nằm ở vị trí `i` (từ 0 tới 25).

Ban đầu ngón tay ở **vị trí 0**. Để gõ một ký tự, ngón tay phải di chuyển tới vị trí của ký tự đó. Thời gian di chuyển từ vị trí `i` tới `j` là `|i - j|`.

Cho chuỗi `word`, tính **tổng thời gian** để gõ nó bằng một ngón tay.

| `keyboard` | `word` | Output | Giải thích |
|---|---|---:|---|
| `"abcdefghijklmnopqrstuvwxyz"` | `"cba"` | `4` | 0→2 (2), 2→1 (1), 1→0 (1) |
| `"pqrstuvwxyzabcdefghijklmno"` | `"leetcode"` | `73` | tính ở phần trace |

**Constraints (theo hiểu biết):**
- `keyboard.length == 26`, chứa mỗi chữ thường tiếng Anh **đúng một lần**.
- `1 <= word.length <= 10^4`, `word[i]` là chữ thường.

In [1]:
from typing import List

keyboard = "abcdefghijklmnopqrstuvwxyz"
word = "cba"
pos = 0
total = 0
for ch in word:
    j = keyboard.index(ch)
    print(f"gõ {ch!r}: từ {pos} tới {j}, mất |{pos}-{j}| = {abs(pos - j)}")
    total += abs(pos - j)
    pos = j
print("tổng =", total)

gõ 'c': từ 0 tới 2, mất |0-2| = 2
gõ 'b': từ 2 tới 1, mất |2-1| = 1
gõ 'a': từ 1 tới 0, mất |1-0| = 1
tổng = 4


## 2. Nền tảng Python cần biết

### 2.1. `abs(x)` — giá trị tuyệt đối

Khoảng cách không âm: `abs(2 - 5) = 3`, `abs(5 - 2) = 3`.

In [2]:
print(abs(2 - 5), abs(5 - 2), abs(0))

3 3 0


### 2.2. `str.index(ch)` và `str.find(ch)`

- `s.index(ch)` trả vị trí **đầu tiên** của `ch`; không có → `ValueError`.
- `s.find(ch)` giống vậy nhưng không có → trả `-1`.
- Cả hai **duyệt chuỗi từ đầu** → `O(len(s))` mỗi lần gọi.

In [3]:
kb = "pqrstuvwxyzabcdefghijklmno"
print(kb.index("p"), kb.index("a"), kb.index("o"))
print(kb.find("A"))         # chữ hoa không có → -1
try:
    kb.index("A")
except ValueError as e:
    print("ValueError:", e)

0 11 25
-1
ValueError: substring not found


### 2.3. `enumerate` và dict comprehension

- `enumerate(s)` sinh các cặp `(index, ký tự)`.
- `{ch: i for i, ch in enumerate(s)}` tạo dict **ký tự → vị trí** chỉ trong một dòng.

In [4]:
kb = "pqrstuvwxyzabcdefghijklmno"
print(list(enumerate(kb))[:5])
pos = {ch: i for i, ch in enumerate(kb)}
print(pos["p"], pos["a"], pos["l"], pos["e"])
print(len(pos))

[(0, 'p'), (1, 'q'), (2, 'r'), (3, 's'), (4, 't')]
0 11 22 15
26


### 2.4. Mảng 26 + `ord`

Vì chỉ có chữ thường, ta có thể dùng list 26 ô: `at[ord(ch) - ord('a')] = vị trí của ch`.

In [5]:
kb = "pqrstuvwxyzabcdefghijklmno"
at = [0] * 26
for i, ch in enumerate(kb):
    at[ord(ch) - ord("a")] = i
print(at)
print("vị trí của 'l' =", at[ord("l") - ord("a")])

[11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
vị trí của 'l' = 22


## 3. Cách 1 — Baseline: `keyboard.index` cho từng ký tự

Duy trì `prev` = vị trí hiện tại của ngón tay (ban đầu 0). Với mỗi ký tự: tìm vị trí `j`, cộng `abs(prev - j)`, rồi `prev = j`.

In [6]:
def calculate_time_index(keyboard: str, word: str) -> int:
    prev = 0
    total = 0
    for ch in word:
        j = keyboard.index(ch)     # duyệt keyboard: O(26)
        total += abs(prev - j)
        prev = j
    return total

print(calculate_time_index("abcdefghijklmnopqrstuvwxyz", "cba"))           # 4
print(calculate_time_index("pqrstuvwxyzabcdefghijklmno", "leetcode"))      # 73

4
73


### Nút thắt
Mỗi `keyboard.index(ch)` tốn tới 26 phép so sánh. Với `word` dài `n`, tổng `O(26·n)`. Vì 26 là hằng số nên vẫn là `O(n)`, nhưng ta đang **tìm đi tìm lại** cùng một thông tin — vị trí của một chữ không bao giờ đổi.

## 4. Insight — Tra trước một lần, dùng mãi

Bố cục bàn phím cố định → lập **bảng tra vị trí** một lần (`26` bước), sau đó mỗi ký tự chỉ tra `O(1)`.

```text
pos = {'p':0, 'q':1, ..., 'z':10, 'a':11, ..., 'o':25}
```

In [7]:
kb = "pqrstuvwxyzabcdefghijklmno"
pos = {ch: i for i, ch in enumerate(kb)}
for ch in "leetcode":
    print(ch, "->", pos[ch], "| index() cũng ra", kb.index(ch))

l -> 22 | index() cũng ra 22
e -> 15 | index() cũng ra 15
e -> 15 | index() cũng ra 15
t -> 4 | index() cũng ra 4
c -> 13 | index() cũng ra 13
o -> 25 | index() cũng ra 25
d -> 14 | index() cũng ra 14
e -> 15 | index() cũng ra 15


## 5. Cách 2 — Lời giải chính: dict vị trí + biến `prev`

**Mổ xẻ:**
1. `pos = {ch: i for i, ch in enumerate(keyboard)}`.
2. `prev = 0`, `total = 0`.
3. Với mỗi `ch` trong `word`: `cur = pos[ch]`, `total += abs(cur - prev)`, `prev = cur`.
4. Trả `total`.

In [8]:
def calculate_time(keyboard: str, word: str) -> int:
    pos = {ch: i for i, ch in enumerate(keyboard)}
    prev = 0
    total = 0
    for ch in word:
        cur = pos[ch]
        total += abs(cur - prev)
        prev = cur
    return total

print(calculate_time("abcdefghijklmnopqrstuvwxyz", "cba"))
print(calculate_time("pqrstuvwxyzabcdefghijklmno", "leetcode"))

4
73


## 6. TRACE từng bước

`keyboard = "pqrstuvwxyzabcdefghijklmno"`, `word = "leetcode"`. Vị trí: l=22, e=15, t=4, c=13, o=25, d=14.

| bước | ký tự | prev | cur | `abs(cur-prev)` | total |
|---:|:---:|---:|---:|---:|---:|
| 1 | l | 0 | 22 | 22 | 22 |
| 2 | e | 22 | 15 | 7 | 29 |
| 3 | e | 15 | 15 | 0 | 29 |
| 4 | t | 15 | 4 | 11 | 40 |
| 5 | c | 4 | 13 | 9 | 49 |
| 6 | o | 13 | 25 | 12 | 61 |
| 7 | d | 25 | 14 | 11 | 72 |
| 8 | e | 14 | 15 | 1 | 73 |

Bước 3 thú vị: gõ lại cùng một chữ → **chi phí 0**.

In [9]:
def trace_calculate_time(keyboard: str, word: str) -> int:
    pos = {ch: i for i, ch in enumerate(keyboard)}
    prev, total = 0, 0
    print(f"{'bước':>4} {'ch':>3} {'prev':>5} {'cur':>4} {'move':>5} {'total':>6}")
    for step, ch in enumerate(word, start=1):
        cur = pos[ch]
        move = abs(cur - prev)
        total += move
        print(f"{step:>4} {ch:>3} {prev:>5} {cur:>4} {move:>5} {total:>6}")
        prev = cur
    return total

trace_calculate_time("pqrstuvwxyzabcdefghijklmno", "leetcode")

bước  ch  prev  cur  move  total
   1   l     0   22    22     22
   2   e    22   15     7     29
   3   e    15   15     0     29
   4   t    15    4    11     40
   5   c     4   13     9     49
   6   o    13   25    12     61
   7   d    25   14    11     72
   8   e    14   15     1     73


73

`enumerate(word, start=1)`: đếm bước từ 1 thay vì 0 — chỉ để in cho dễ đọc. Định dạng `f"{x:>5}"` căn phải trong 5 ký tự để bảng thẳng cột.

## 7. Cách 3 — Mảng 26 thay cho dict

In [10]:
def calculate_time_array(keyboard: str, word: str) -> int:
    at = [0] * 26
    for i, ch in enumerate(keyboard):
        at[ord(ch) - 97] = i
    prev = total = 0
    for ch in word:
        cur = at[ord(ch) - 97]
        total += abs(cur - prev)
        prev = cur
    return total

print(calculate_time_array("pqrstuvwxyzabcdefghijklmno", "leetcode"))

73


## 8. Cách 4 — Viết kiểu "cặp liên tiếp" với `zip`

Chèn vị trí bắt đầu 0 vào đầu dãy vị trí, rồi cộng khoảng cách giữa mọi **cặp liền kề**:

```text
positions = [0, 22, 15, 15, 4, 13, 25, 14, 15]
cặp: (0,22), (22,15), (15,15), ...
```

`zip(a, a[1:])` ghép phần tử thứ `i` với phần tử thứ `i+1`.

In [11]:
a = [0, 22, 15, 15]
print(list(zip(a, a[1:])))

def calculate_time_zip(keyboard: str, word: str) -> int:
    pos = {ch: i for i, ch in enumerate(keyboard)}
    positions = [0] + [pos[ch] for ch in word]
    return sum(abs(y - x) for x, y in zip(positions, positions[1:]))

print(calculate_time_zip("pqrstuvwxyzabcdefghijklmno", "leetcode"))

[(0, 22), (22, 15), (15, 15)]
73


Cách này đẹp nhưng tạo list `positions` dài `n + 1` → bộ nhớ `O(n)`. Bản dùng `prev` chỉ `O(1)` (ngoài bảng 26).

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def calculateTime(self, keyboard: str, word: str) -> int:
        pos = {ch: i for i, ch in enumerate(keyboard)}
        prev = 0
        total = 0
        for ch in word:
            cur = pos[ch]
            total += abs(cur - prev)
            prev = cur
        return total

sol = Solution()
print(sol.calculateTime("abcdefghijklmnopqrstuvwxyz", "cba"))
print(sol.calculateTime("pqrstuvwxyzabcdefghijklmno", "leetcode"))

4
73


In [13]:
class SolutionZip:
    def calculateTime(self, keyboard: str, word: str) -> int:
        pos = {ch: i for i, ch in enumerate(keyboard)}
        p = [0] + [pos[ch] for ch in word]
        return sum(abs(b - a) for a, b in zip(p, p[1:]))

print(SolutionZip().calculateTime("pqrstuvwxyzabcdefghijklmno", "leetcode"))

73


## 10. Vì sao đúng?

Ngón tay gõ lần lượt `word[0], word[1], ...`. Tổng thời gian = tổng các đoạn di chuyển:

```text
|p(w0) - 0| + |p(w1) - p(w0)| + ... + |p(w_{n-1}) - p(w_{n-2})|
```

**Invariant:** trước khi xử lý ký tự thứ `k`, `prev` = vị trí ngón tay sau khi gõ `k` ký tự đầu (với `k = 0` là vị trí 0), và `total` = tổng thời gian đã dùng cho `k` ký tự đó. Mỗi vòng cộng đúng một đoạn và cập nhật `prev` → invariant giữ cho `k + 1`. Hết vòng: `total` là đáp án.

Bảng `pos` đúng vì `keyboard` chứa mỗi chữ đúng một lần (ánh xạ một-một).

## 11. Độ phức tạp

Gọi `n = len(word)`, `K = 26` (độ dài keyboard).

| Cách | Time | Space |
|---|---|---|
| `keyboard.index` | `O(K·n)` = `O(26n)` | `O(1)` |
| dict + `prev` | `O(K + n)` | `O(K)` = `O(1)` |
| mảng 26 | `O(K + n)` | `O(1)` |
| `zip` | `O(K + n)` | `O(n)` |

## 12. Bộ test: mẫu, biên, phản ví dụ, đối chiếu

In [14]:
ABC = "abcdefghijklmnopqrstuvwxyz"
cases = [
    (ABC, "cba", 4),
    ("pqrstuvwxyzabcdefghijklmno", "leetcode", 73),
    (ABC, "a", 0),                      # đứng sẵn ở vị trí 0
    (ABC, "z", 25),
    (ABC, "aaaa", 0),                   # gõ lặp cùng phím
    (ABC, "az", 25),
    (ABC, "zaz", 75),                   # 25 + 25 + 25
    (ABC[::-1], "a", 25),               # bàn phím đảo ngược
    (ABC, "abcdefghijklmnopqrstuvwxyz", 25),
]
solvers = [calculate_time_index, calculate_time, calculate_time_array, calculate_time_zip,
           Solution().calculateTime, SolutionZip().calculateTime]
for kb, w, expected in cases:
    for f in solvers:
        assert f(kb, w) == expected, (f, kb, w)
    print(f"kb={kb[:6]}…  word={w[:12]!r:<15} -> {expected} PASS")

kb=abcdef…  word='cba'           -> 4 PASS
kb=pqrstu…  word='leetcode'      -> 73 PASS
kb=abcdef…  word='a'             -> 0 PASS
kb=abcdef…  word='z'             -> 25 PASS
kb=abcdef…  word='aaaa'          -> 0 PASS
kb=abcdef…  word='az'            -> 25 PASS
kb=abcdef…  word='zaz'           -> 75 PASS
kb=zyxwvu…  word='a'             -> 25 PASS
kb=abcdef…  word='abcdefghijkl'  -> 25 PASS


In [15]:
import random
random.seed(1165)
for _ in range(300):
    letters = list(ABC)
    random.shuffle(letters)
    kb = "".join(letters)
    w = "".join(random.choice(ABC) for _ in range(random.randint(1, 50)))
    expected = calculate_time_index(kb, w)
    for f in solvers[1:]:
        assert f(kb, w) == expected
# word dài tối đa
big = "".join(random.choice(ABC) for _ in range(10**4))
print("word dài 10^4:", Solution().calculateTime(ABC, big))
print("300 bàn phím ngẫu nhiên khớp baseline. PASS")

word dài 10^4: 86837
300 bàn phím ngẫu nhiên khớp baseline. PASS


`random.shuffle(lst)` xáo trộn list **tại chỗ** — dùng để sinh bố cục bàn phím ngẫu nhiên hợp lệ (vẫn đủ 26 chữ, mỗi chữ một lần).

## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên vị trí bắt đầu là 0 (bỏ qua bước di chuyển đầu tiên)
```python
prev = pos[word[0]]          # bỏ mất |pos[word[0]] - 0|
```

### Lỗi 2: Không cập nhật `prev` → luôn đo từ vị trí 0.

### Lỗi 3: Quên `abs` → khoảng cách âm làm tổng nhỏ đi.

### Lỗi 4: Dùng `word.index` thay vì `keyboard.index` (tra nhầm chuỗi).

In [16]:
def wrong_start(kb, w):
    pos = {ch: i for i, ch in enumerate(kb)}
    prev, total = pos[w[0]], 0
    for ch in w:
        total += abs(pos[ch] - prev)
        prev = pos[ch]
    return total

def wrong_no_update(kb, w):
    pos = {ch: i for i, ch in enumerate(kb)}
    return sum(abs(pos[ch] - 0) for ch in w)

def wrong_no_abs(kb, w):
    pos = {ch: i for i, ch in enumerate(kb)}
    prev, total = 0, 0
    for ch in w:
        total += pos[ch] - prev
        prev = pos[ch]
    return total

kb, w = "abcdefghijklmnopqrstuvwxyz", "cba"
print("đúng:", Solution().calculateTime(kb, w))
print("quên vị trí 0:", wrong_start(kb, w))
print("không cập nhật prev:", wrong_no_update(kb, w))
print("quên abs:", wrong_no_abs(kb, w))

đúng: 4
quên vị trí 0: 2
không cập nhật prev: 3
quên abs: 0


Quên `abs` cho ra `0` với `"cba"` vì tổng có dấu "co rút" (telescoping) về `vị trí cuối - 0 = 0`. Đây là một phản ví dụ rất đáng nhớ.

## 14. Mẹo nhận dạng pattern

- Có **bố cục cố định** + nhiều lần hỏi "chữ X ở đâu?" → **lập bảng tra một lần** (dict/mảng).
- Tổng chi phí "đi từ điểm này sang điểm kia" theo thứ tự → giữ biến **`prev`** và cộng dồn `cost(prev, cur)`.
- Khoảng cách trên một đường thẳng = `abs(a - b)`; trên lưới = `abs(dx) + abs(dy)` (Manhattan); trên vòng tròn = `min(d, n - d)`.

Bài liên quan: 1184 (Distance Between Bus Stops), 1320 (Minimum Distance to Type a Word Using Two Fingers), 1266 (Minimum Time Visiting All Points).

## 15. Biến thể

### 15.1. Bàn phím dạng lưới (nhiều hàng)
Mỗi phím có tọa độ `(hàng, cột)`, chi phí là khoảng cách Manhattan. Ví dụ bàn phím 5 cột: chữ thứ `i` ở `(i // 5, i % 5)`.

In [17]:
def calculate_time_grid(keyboard: str, word: str, cols: int) -> int:
    pos = {ch: divmod(i, cols) for i, ch in enumerate(keyboard)}   # (hàng, cột)
    pr, pc = 0, 0
    total = 0
    for ch in word:
        r, c = pos[ch]
        total += abs(r - pr) + abs(c - pc)
        pr, pc = r, c
    return total

print(divmod(7, 5))   # (1, 2): hàng 1, cột 2
print(calculate_time_grid("abcdefghijklmnopqrstuvwxyz", "cba", 5))   # cùng hàng 0 → như bản 1 hàng: 4
print(calculate_time_grid("abcdefghijklmnopqrstuvwxyz", "af", 5))    # a(0,0) → f(1,0): 1

(1, 2)
4
1


### 15.2. Bàn phím vòng tròn
Nếu đi được vòng quanh (phím 25 cạnh phím 0), chi phí giữa `i` và `j` là `min(d, 26 - d)` với `d = abs(i - j)`.

In [18]:
def calculate_time_circular(keyboard: str, word: str) -> int:
    pos = {ch: i for i, ch in enumerate(keyboard)}
    n = len(keyboard)
    prev = total = 0
    for ch in word:
        d = abs(pos[ch] - prev)
        total += min(d, n - d)
        prev = pos[ch]
    return total

print(calculate_time_circular("abcdefghijklmnopqrstuvwxyz", "z"))     # 1 thay vì 25
print(calculate_time_circular("abcdefghijklmnopqrstuvwxyz", "zaz"))   # 1 + 1 + 1

1
3


### 15.3. So sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| `index` | tìm lại mỗi lần | `O(26n)` | `O(1)` | ngắn, chấp nhận được |
| dict + prev | bảng tra + trạng thái | `O(26 + n)` | `O(1)` | **nộp bài** |
| mảng 26 | bảng tra bằng `ord` | `O(26 + n)` | `O(1)` | nhanh hằng số |
| zip cặp liền kề | cộng khoảng cách cặp | `O(26 + n)` | `O(n)` | đẹp, tốn bộ nhớ |

## 16. Bài tập vận dụng (TODO)

### Bài 1 — Tính tay
`keyboard = "abcdefghijklmnopqrstuvwxyz"`, `word = "hello"`. Tổng thời gian?

### Bài 2 — Lập bảng vị trí bằng vòng `for` (không dùng comprehension)

In [19]:
def build_pos_exercise(keyboard: str) -> dict:
    # TODO: pos = {}; for i, ch in enumerate(keyboard): ...
    pass

# assert build_pos_exercise("bac") == {"b": 0, "a": 1, "c": 2}

### Bài 3 — Hoàn thiện lời giải với `prev`

In [20]:
def calculate_time_exercise(keyboard: str, word: str) -> int:
    # TODO 1: bảng vị trí
    # TODO 2: prev = 0, total = 0
    # TODO 3: cộng abs(cur - prev), cập nhật prev
    pass

# assert calculate_time_exercise("abcdefghijklmnopqrstuvwxyz", "cba") == 4
# assert calculate_time_exercise("pqrstuvwxyzabcdefghijklmno", "leetcode") == 73

### Bài 4 — Đoạn di chuyển dài nhất
Trả về độ dài lớn nhất của **một** lần di chuyển khi gõ `word` (tính cả bước đầu từ vị trí 0).

In [21]:
def longest_move_exercise(keyboard: str, word: str) -> int:
    # TODO
    pass

# assert longest_move_exercise("pqrstuvwxyzabcdefghijklmno", "leetcode") == 22

### Bài 5 — Bàn phím có thể thiếu chữ
Nếu `word` có ký tự không nằm trên bàn phím, trả `-1`.

In [22]:
def calculate_time_safe_exercise(keyboard: str, word: str) -> int:
    # TODO: kiểm tra ch in pos trước khi tra
    pass

# assert calculate_time_safe_exercise("abc", "cab") == 2 + 2 + 1
# assert calculate_time_safe_exercise("abc", "abd") == -1

### Bài 6 — Mini challenge: chọn vị trí bắt đầu tốt nhất
Nếu được chọn vị trí bắt đầu `s` tùy ý (0..25), tổng thời gian nhỏ nhất là bao nhiêu?

In [23]:
def best_start_exercise(keyboard: str, word: str) -> int:
    # TODO: chỉ có bước đầu phụ thuộc s → chọn s = vị trí word[0]
    pass

# assert best_start_exercise("abcdefghijklmnopqrstuvwxyz", "cba") == 2
# assert best_start_exercise("pqrstuvwxyzabcdefghijklmno", "leetcode") == 73 - 22

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | vị trí = thứ tự chữ cái | h=7, e=4, l=11, o=14 | 7 + 3 + 7 + 0 + 3 |
| 2 | dict rỗng | `enumerate` | `pos[ch] = i` |
| 3 | xem mục 5 | biến `prev` | `total += abs(cur - prev); prev = cur` |
| 4 | như bài chính nhưng lấy max | `best = 0` | `best = max(best, abs(cur - prev))` |
| 5 | `in` kiểm tra key | trả sớm `-1` | `if ch not in pos: return -1` |
| 6 | bước đầu có thể = 0 | tổng gốc trừ bước đầu | `total - pos[word[0]]` |

## 18. Đáp án tham khảo

**Bài 1:** `0→h(7)`: 7, `h→e(4)`: 3, `e→l(11)`: 7, `l→l`: 0, `l→o(14)`: 3 → **20**.

In [24]:
def build_pos_answer(keyboard: str) -> dict:
    pos = {}
    for i, ch in enumerate(keyboard):
        pos[ch] = i
    return pos

def calculate_time_answer(keyboard: str, word: str) -> int:
    pos = build_pos_answer(keyboard)
    prev = total = 0
    for ch in word:
        cur = pos[ch]
        total += abs(cur - prev)
        prev = cur
    return total

def longest_move_answer(keyboard: str, word: str) -> int:
    pos = build_pos_answer(keyboard)
    prev = best = 0
    for ch in word:
        best = max(best, abs(pos[ch] - prev))
        prev = pos[ch]
    return best

def calculate_time_safe_answer(keyboard: str, word: str) -> int:
    pos = build_pos_answer(keyboard)
    prev = total = 0
    for ch in word:
        if ch not in pos:
            return -1
        total += abs(pos[ch] - prev)
        prev = pos[ch]
    return total

def best_start_answer(keyboard: str, word: str) -> int:
    pos = build_pos_answer(keyboard)
    return calculate_time_answer(keyboard, word) - pos[word[0]]

assert Solution().calculateTime("abcdefghijklmnopqrstuvwxyz", "hello") == 20
assert build_pos_answer("bac") == {"b": 0, "a": 1, "c": 2}
assert calculate_time_answer("abcdefghijklmnopqrstuvwxyz", "cba") == 4
assert calculate_time_answer("pqrstuvwxyzabcdefghijklmno", "leetcode") == 73
assert longest_move_answer("pqrstuvwxyzabcdefghijklmno", "leetcode") == 22
assert calculate_time_safe_answer("abc", "cab") == 5
assert calculate_time_safe_answer("abc", "abd") == -1
assert best_start_answer("abcdefghijklmnopqrstuvwxyz", "cba") == 2
assert best_start_answer("pqrstuvwxyzabcdefghijklmno", "leetcode") == 51
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def calculateTime(self, keyboard: str, word: str) -> int:
        pos = {ch: i for i, ch in enumerate(keyboard)}
        prev = total = 0
        for ch in word:
            total += abs(pos[ch] - prev)
            prev = pos[ch]
        return total
```

**Checklist:**
- [ ] Ngón tay **bắt đầu ở vị trí 0** (bước đầu tốn `pos[word[0]]`)?
- [ ] Dùng `abs` cho khoảng cách?
- [ ] Cập nhật `prev` sau mỗi ký tự?
- [ ] Lập bảng vị trí **một lần**, không gọi `index` lặp lại (nếu muốn tối ưu)?
- [ ] Test chữ lặp liên tiếp (chi phí 0) và chữ ở vị trí 0?